# 03 - Quantum-Classical Hybrid (Track C)

**Purpose.** Attach a trainable variational quantum circuit (4 qubits, angle encoding, Rot layers, cyclic CNOT, Pauli-Z readout) to the best classical backbone (selected on validation) and compare it with parameter-matched classical controls under one predetermined budget, over several seeds. Record simulator cost and trainability diagnostics.

**Requires (inputs).** Notebook 00 output (manifest) and notebook 01/02 outputs (trained backbone checkpoint), same MODE.

**Produces (outputs, under `/kaggle/working/rsna_ich_artifacts/<MODE>/`).** `features/<backbone>/features_<split>.npz`, `experiments/C_<head>_s<seed>/`, `analysis/trackC_*.csv`, `analysis/vqc_circuit.txt`, `analysis/simulator_benchmark.csv`.

> Research software. Outputs are model predictions for a research study and **not** clinical diagnoses.
> Results in this notebook are only valid for the MODE they were produced in (`smoke` results check correctness and say nothing about performance).

In [1]:
# ==== Setup: locate the project code and make `src` importable ====================================
# Works on Kaggle (project code attached as a Dataset, possibly nested after zip extraction) and locally.
import os, sys, importlib
from pathlib import Path

def find_project_root(max_depth: int = 5) -> Path:
    def ok(p: Path) -> bool:
        return (p / "src" / "__init__.py").is_file() and (p / "configs" / "data_config.yaml").is_file()
    cands = []
    if os.environ.get("RSNA_ICH_PROJECT_ROOT"):
        cands.append(Path(os.environ["RSNA_ICH_PROJECT_ROOT"]))
    cands += [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/RSNA_ICH_Research")]
    for c in cands:
        if ok(c):
            return c.resolve()
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        stack = [(base, 0)]
        while stack:
            d, depth = stack.pop()
            if ok(d):
                return d.resolve()
            if depth < max_depth:
                try:
                    stack += [(p, depth + 1) for p in d.iterdir() if p.is_dir() and not p.name.startswith("stage_2_")]
                except OSError:
                    pass
    raise FileNotFoundError(
        "Project code not found. On Kaggle: 'Add Input' -> the dataset containing RSNA_ICH_Research "
        "(with src/ and configs/). Locally: run the notebook from RSNA_ICH_Research/notebooks or set "
        "the RSNA_ICH_PROJECT_ROOT environment variable.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for _m in ["src", "src.utils.context", "src.data", "src.models", "src.quantum", "src.training", "src.evaluation"]:
    importlib.import_module(_m)  # fails loudly here if a dependency is missing
import src
print(f"Project root: {PROJECT_ROOT}  |  src version {src.__version__}")

Project root: /kaggle/input/datasets/mehulkhannaa/rsna-ich-research-code/RSNA_ICH_Research  |  src version 1.0.0


In [2]:
# ==== Optional dependencies ======================================================================
# Kaggle images ship torch, timm, pydicom, scikit-learn and OpenCV. PennyLane is only needed for the
# reference quantum backend (the default torch simulator does not need it). With internet enabled:
!pip install -q pennylane
# Without internet, attach a dataset of wheels and run: !pip install --no-index --find-links /kaggle/input/<wheels> pennylane
from src.utils.runtime import package_versions
print(package_versions())

{'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'timm': '1.0.30', 'pennylane': '0.45.1', 'pydicom': '3.0.2', 'numpy': '2.1.3', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'yaml': '6.0.3', 'cv2': '4.14.0'}


In [3]:
# ==== Configuration ================================================================================
# MODE: "smoke" (correctness check, minutes) | "pilot" (representative subset) | "full" (largest feasible).
MODE = "pilot"
# Optional per-config overrides without editing YAML, e.g. {"transformer": {"train": {"epochs": 2}}}
OVERRIDES = {}
import json
if os.environ.get("RSNA_ICH_OVERRIDES_JSON"):          # test hook used by tools/run_notebooks_synthetic.py
    OVERRIDES = json.loads(os.environ["RSNA_ICH_OVERRIDES_JSON"])
MODE = os.environ.get("RSNA_ICH_MODE", MODE)

from src.utils.context import ProjectContext
ctx = ProjectContext(PROJECT_ROOT, MODE, OVERRIDES)
print(ctx.describe())
print("Installed packages:", ctx.runtime.packages)
ctx.save_run_info("03_quantum")

ProjectContext(mode=pilot, root=/kaggle/input/datasets/mehulkhannaa/rsna-ich-research-code/RSNA_ICH_Research, output=/kaggle/working/rsna_ich_artifacts/pilot)
  GPU: Tesla T4 (15.6 GB), Tesla T4 (15.6 GB) | CUDA=True bf16=False | CPUs=4 | RAM avail=31.66 GB | free disk (/kaggle/working/rsna_ich_artifacts)=20.94 GB
Installed packages: {'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'timm': '1.0.30', 'pennylane': '0.45.1', 'pydicom': '3.0.2', 'numpy': '2.1.3', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'yaml': '6.0.3', 'cv2': '4.14.0'}


PosixPath('/kaggle/working/rsna_ich_artifacts/pilot/run_info/03_quantum.json')

In [4]:
# ==== Unpack experiments from zipped notebook outputs (Kaggle zips outputs with many files) ====
import glob, zipfile, shutil
key = f"{MODE}/experiments/"
dst_root = ctx.output_root / "experiments"
zips = [z for pat in ["/kaggle/input/*/_output_.zip", "/kaggle/input/*/*/_output_.zip",
                      "/kaggle/input/*/*/*/_output_.zip", "/kaggle/input/*/*/*/*/_output_.zip"]
        for z in glob.glob(pat)]
print("Zipped inputs:", zips)
n = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        for name in zf.namelist():
            if key not in name or name.endswith("/") or name.endswith("last.pt"):
                continue
            rel = name.split(key, 1)[1]
            if not rel.startswith(("A_", "B_", "H_")):
                continue
            out = dst_root / rel
            if out.exists():
                continue
            out.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(name) as src, open(out, "wb") as f:
                shutil.copyfileobj(src, f)
            n += 1
print(f"Extracted {n} files")
print(sorted(p.name for p in dst_root.iterdir() if p.is_dir()))

Zipped inputs: ['/kaggle/input/notebooks/mehulkhannaa/01-classical-transformer-benchmark/_output_.zip']
Extracted 62 files
['A_deit_small', 'A_maxvit_tiny', 'A_swin_tiny', 'H_maxvit_tiny_brain3']


In [5]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src import LABELS
from src.utils.io import atomic_write_json, atomic_save_dataframe, read_json, dataframe_to_markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## 1. Manifest, configuration and backbone selection (validation only)

In [6]:
from src.training.experiment import select_best_by_validation, load_model_from_result, split_frames, extract_features, run_head_experiment
manifest, man_info = ctx.manifest()
ctx.restore_experiments(("C_",))
qcfg = ctx.cfg["quantum"]
src_cfg = qcfg["backbone"]["source"]
if src_cfg == "auto":
    bb = select_best_by_validation(ctx, tuple(qcfg["backbone"]["candidates_prefix"]))
else:
    r = ctx.results((src_cfg,)); bb = r.iloc[0].to_dict() if len(r) else None
if bb is None:
    raise RuntimeError("No trained backbone found. Run/attach notebook 01 (and/or 02) for this MODE first.")
print("Backbone:", bb["experiment_id"], "| val macro AUC", round(bb["val_metrics"]["macro_roc_auc"], 4),
      "| weights", bb["build"]["status"])

Backbone: A_maxvit_tiny | val macro AUC 0.9577 | weights pretrained


## 2. Extract backbone features once (frozen backbone; restartable cache)
The same per-split frames as the backbone's own evaluation are used, so all heads see identical images.

In [7]:
from src.data.preprocessing import PreprocessConfig
backbone = load_model_from_result(bb, man_info["manifest_hash"])
frames = split_frames(manifest, bb["config"]["limits"], bb["config"]["seed"])
pp = PreprocessConfig.from_dict(bb["config"]["preprocess"])
feat_dir = ctx.dir("features", bb["experiment_id"])
source_fp = f'{bb["experiment_id"]}:{bb["config_fingerprint"]}'
feats = extract_features(ctx, backbone, frames, pp, feat_dir,
                         {"source_fingerprint": source_fp, "manifest_hash": man_info["manifest_hash"]},
                         batch_size=qcfg["backbone"]["feature_batch_size"])
print({s: v["features"].shape for s, v in feats.items()})
del backbone; torch.cuda.empty_cache() if torch.cuda.is_available() else None

[features] train: extracted (42889, 512) in 1955s
[features] val: extracted (8618, 512) in 399s
[features] test: extracted (9552, 512) in 437s
{'train': (42889, 512), 'val': (8618, 512), 'test': (9552, 512)}


## 3. The circuit, its resources and a backend cross-check
Two exact simulators are available: a batched PyTorch state-vector simulator (default) and PennyLane. They must agree; any disagreement stops the notebook.

In [8]:
from src.quantum import VQCConfig, VQCLayer, circuit_resources, benchmark_simulator
vdef = {**qcfg["head_defaults"]["vqc"], "n_qubits": qcfg["head_defaults"]["k"]}
print(circuit_resources(VQCConfig(**vdef)))
crosscheck = {"enabled": qcfg["pennylane_crosscheck"]["enabled"]}
try:
    import pennylane as qml
    lt = VQCLayer(VQCConfig(**{**vdef, "backend": "torch"}))
    lp = VQCLayer(VQCConfig(**{**vdef, "backend": "pennylane"})); lp.weights.data = lt.weights.data.clone()
    print(lp.draw()); (ctx.dir("analysis") / "vqc_circuit.txt").write_text(lp.draw(), encoding="utf-8")
    x = torch.randn(qcfg["pennylane_crosscheck"]["n_samples"], vdef["n_qubits"])
    diff = float((lt(x) - lp(x)).abs().max())
    crosscheck.update(pennylane=qml.__version__, max_abs_diff=diff)
    assert diff < 1e-4, f"backends disagree: {diff}"
    bench_rows = benchmark_simulator(vdef, (64, 256)) + benchmark_simulator({**vdef, "backend": "pennylane"}, (64,), repeats=2)
except ImportError:
    crosscheck.update(pennylane="not installed", note="cross-check skipped; torch backend unit-tested against PennyLane offline")
    bench_rows = benchmark_simulator(vdef, (64, 256))
print(crosscheck)
bench = pd.DataFrame(bench_rows); display(bench)
atomic_save_dataframe(bench, ctx.dir("analysis") / "simulator_benchmark.csv")
atomic_write_json(ctx.dir("analysis") / "vqc_crosscheck.json", crosscheck)

{'n_qubits': 4, 'n_layers': 2, 'trainable_params': 24, 'encoding_gates': 4, 'rotation_gates': 8, 'cnot_gates': 8, 'hadamard_gates': 0, 'state_dim': 16, 'measurements': 4}
0: ──RY(M0)──Rot(6.06,3.81,0.84)─╭●───────╭X──Rot(5.28,3.03,1.38)─╭●───────╭X─┤  <Z>
1: ──RY(M0)──Rot(3.22,1.22,3.53)─╰X─╭●────│───Rot(2.73,0.53,5.32)─╰X─╭●────│──┤  <Z>
2: ──RY(M0)──Rot(4.81,2.74,0.37)────╰X─╭●─│───Rot(4.80,1.01,3.72)────╰X─╭●─│──┤  <Z>
3: ──RY(M0)──Rot(2.91,2.27,1.68)───────╰X─╰●──Rot(3.36,4.24,2.25)───────╰X─╰●─┤  <Z>

M0 = 
tensor([0.], dtype=torch.float64)


/tmp/ipykernel_23/1774517850.py:11: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  diff = float((lt(x) - lp(x)).abs().max())


{'enabled': True, 'pennylane': '0.45.1', 'max_abs_diff': 3.5762786865234375e-07}


,backend,device,batch,fwd_bwd_ms,us_per_sample,n_qubits,n_layers,trainable_params,encoding_gates,rotation_gates,cnot_gates,hadamard_gates,state_dim,measurements
0,torch,cuda,64,25.601296,400.020256,4,2,24,4,8,8,0,16,4
1,torch,cuda,256,24.170991,94.417932,4,2,24,4,8,8,0,16,4
2,pennylane,cpu,64,25.021801,390.965641,4,2,24,4,8,8,0,16,4


PosixPath('/kaggle/working/rsna_ich_artifacts/pilot/analysis/vqc_crosscheck.json')

## 4. Predetermined head grid x seeds (primary comparison + bounded ablations)
| id | role |
|---|---|
| `linear_probe` | upper reference: Linear(d->6) on the same features |
| `bottleneck_identity` | control: same k-dim encoder + readout, no processing block |
| `bottleneck_mlp` | control: VQC replaced by Linear(k,k)+tanh (parameter-matched) |
| `vqc_q4_l2_cyclic` | quantum-classical hybrid (Paper-2 circuit) |
| ablations | no entanglement (D), depth 1/4 (E), qubits 2/8 (F) + k=8 classical control |

In [9]:
import copy
def head_cfg(spec):
    d = copy.deepcopy(qcfg["head_defaults"])
    d["kind"] = spec["kind"]
    if "k" in spec: d["k"] = spec["k"]
    d["vqc"] = {**d["vqc"], **spec.get("vqc", {})}
    if spec["kind"] != "vqc": d.pop("vqc")
    return d
grid = list(qcfg["primary"]) + (list(qcfg["ablations"]) if qcfg["run_ablations"] else [])
print(f"{len(grid)} head configurations x {len(qcfg['seeds'])} seeds = {len(grid) * len(qcfg['seeds'])} runs")
head_results = []
for spec in grid:
    for seed in qcfg["seeds"]:
        exp_id = f"C_{spec['id']}_s{seed}"
        r = run_head_experiment(ctx, exp_id, feats, head_cfg(spec), qcfg["train"], qcfg["loss"], frames=frames,
                                manifest_info=man_info, source_fingerprint=source_fp, seed=seed,
                                batch_size=qcfg["batch_size"], threshold_method=qcfg["threshold_method"], n_boot=qcfg["n_boot"])
        head_results.append({"head": spec["id"], "seed": seed, "status": r["status"],
                             "val_macro_auc": r.get("val_metrics", {}).get("macro_roc_auc", np.nan),
                             "test_macro_auc": r.get("test_metrics", {}).get("macro_roc_auc", np.nan),
                             "test_macro_ap": r.get("test_metrics", {}).get("macro_ap", np.nan),
                             "params": (r.get("head_params") or {}).get("total"),
                             "block_params": (r.get("head_params") or {}).get("block"),
                             "train_s": r.get("seconds_total"), "vqc_sim_s": r.get("vqc_sim_seconds"),
                             "epochs": (r.get("fit") or {}).get("epochs_run"), "error": r.get("error")})
hr = pd.DataFrame(head_results)
atomic_save_dataframe(hr, ctx.dir("analysis") / "trackC_runs.csv")
display(hr)

10 head configurations x 3 seeds = 30 runs
[epoch 1/30] train_loss=0.3815 val_loss=0.2428 val_macro_auc=0.9562 * (1s)
[epoch 2/30] train_loss=0.1909 val_loss=0.2401 val_macro_auc=0.9578 * (0s)
[epoch 3/30] train_loss=0.1852 val_loss=0.2403 val_macro_auc=0.9579  (0s)
[epoch 4/30] train_loss=0.1823 val_loss=0.2411 val_macro_auc=0.9582 * (0s)
[epoch 5/30] train_loss=0.1799 val_loss=0.2404 val_macro_auc=0.9583 * (0s)
[epoch 6/30] train_loss=0.1783 val_loss=0.2418 val_macro_auc=0.9583  (0s)
[epoch 7/30] train_loss=0.1766 val_loss=0.2435 val_macro_auc=0.9574  (0s)
[epoch 8/30] train_loss=0.1755 val_loss=0.2420 val_macro_auc=0.9576  (1s)
[epoch 9/30] train_loss=0.1744 val_loss=0.2420 val_macro_auc=0.9573  (0s)
[epoch 10/30] train_loss=0.1732 val_loss=0.2428 val_macro_auc=0.9573  (0s)
[trainer] early stopping at epoch 10 (no improvement for 5 epochs)
[epoch 1/30] train_loss=0.3800 val_loss=0.2443 val_macro_auc=0.9556 * (0s)
[epoch 2/30] train_loss=0.1908 val_loss=0.2394 val_macro_auc=0.9574 * 

,head,seed,status,val_macro_auc,test_macro_auc,test_macro_ap,params,block_params,train_s,vqc_sim_s,epochs,error
0,linear_probe,0,succeeded,0.958341,0.915076,0.592651,3078,NaN,43.832232,0.000000,10,None
1,linear_probe,1,succeeded,0.957831,0.915163,0.594876,3078,NaN,42.307833,0.000000,8,None
2,linear_probe,2,succeeded,0.957811,0.914033,0.594048,3078,NaN,42.889147,0.000000,9,None
3,bottleneck_identity,0,succeeded,0.952732,0.918545,0.512995,2242,0.0,43.427294,0.000000,9,None
4,bottleneck_identity,1,succeeded,0.951806,0.904252,0.505806,2242,0.0,44.436162,0.000000,11,None
5,bottleneck_identity,2,succeeded,0.953023,0.917801,0.547253,2242,0.0,43.878856,0.000000,10,None
6,bottleneck_mlp,0,succeeded,0.947749,0.909071,0.484372,2262,20.0,47.748851,0.000000,16,None
7,bottleneck_mlp,1,succeeded,0.950387,0.912892,0.502706,2262,20.0,48.722226,0.000000,17,None
8,bottleneck_mlp,2,succeeded,0.950977,0.917151,0.493993,2262,20.0,46.288348,0.000000,14,None
9,vqc_q4_l2_cyclic,0,succeeded,0.937817,0.903902,0.426393,2266,24.0,62.354088,8.259684,9,None


## 5. Summary across seeds and paired comparisons
Seed-level summary (mean, SD, t-interval) and paired tests across seeds; plus a patient-level paired bootstrap on the held-out test set using the seed-averaged predictions of each head.

In [10]:
from src.evaluation.statistics import seed_summary, paired_seed_test, paired_bootstrap, holm
from src.evaluation.predictions import load_predictions, align, arrays
summ = []
for h, g in hr[hr["status"] == "succeeded"].groupby("head", sort=False):
    sv, st = seed_summary(g["val_macro_auc"]), seed_summary(g["test_macro_auc"])
    summ.append({"head": h, "n_seeds": st["n"], "val_mean": sv["mean"], "val_sd": sv["sd"], "test_mean": st["mean"],
                 "test_sd": st["sd"], "test_ci_low": st["ci_low"], "test_ci_high": st["ci_high"],
                 "params": int(g["params"].iloc[0]) if g["params"].notna().any() else None,
                 "mean_train_s": g["train_s"].mean(), "mean_vqc_sim_s": g["vqc_sim_s"].mean()})
summary = pd.DataFrame(summ); display(summary.round(4))
atomic_save_dataframe(summary, ctx.dir("analysis") / "trackC_summary.csv")

def seed_avg_preds(head):
    dfs = [load_predictions(ctx.dir("experiments", f"C_{head}_s{s}") / "predictions_test.parquet", man_info["manifest_hash"])[0]
           for s in qcfg["seeds"] if (ctx.dir("experiments", f"C_{head}_s{s}") / "predictions_test.parquet").exists()]
    if not dfs: return None
    ids, al = align({i: d for i, d in enumerate(dfs)})
    out = al[0].copy()
    probs = np.mean([arrays(d)[0] for d in al.values()], axis=0)
    for j, l in enumerate(LABELS): out[f"prob_{l}"] = probs[:, j]
    return out

QUANTUM = "vqc_q4_l2_cyclic"
comps = []
qa = seed_avg_preds(QUANTUM)
for ctrl in ["bottleneck_mlp", "bottleneck_identity", "linear_probe"]:
    ca = seed_avg_preds(ctrl)
    a, b = hr[(hr["head"] == QUANTUM) & (hr["status"] == "succeeded")], hr[(hr["head"] == ctrl) & (hr["status"] == "succeeded")]
    m = a.merge(b, on="seed", suffixes=("_q", "_c"))
    st = paired_seed_test(m["test_macro_auc_q"], m["test_macro_auc_c"]) if len(m) else {}
    rec = {"comparison": f"{QUANTUM} - {ctrl}", **{f"seeds_{k}": v for k, v in st.items()}}
    if qa is not None and ca is not None:
        ids, al = align({"q": qa, "c": ca})
        pq, t = arrays(al["q"]); pc_, _ = arrays(al["c"])
        rec.update({f"boot_{k}": v for k, v in paired_bootstrap(t, pq, pc_, al["q"]["group_id"].to_numpy(), "macro_auc",
                                                                 n_boot=max(qcfg["n_boot"], 100), seed=0).items() if k != "method"})
    comps.append(rec)
cmp_df = pd.DataFrame(comps)
if "boot_p_value" in cmp_df: cmp_df["boot_p_holm"] = holm(cmp_df["boot_p_value"])
display(cmp_df.round(4))
atomic_save_dataframe(cmp_df, ctx.dir("analysis") / "trackC_paired.csv")

,head,n_seeds,val_mean,val_sd,test_mean,test_sd,test_ci_low,test_ci_high,params,mean_train_s,mean_vqc_sim_s
0,linear_probe,3,0.9580,0.0003,0.9148,0.0006,0.9132,0.9163,3078,43.0097,0.0000
1,bottleneck_identity,3,0.9525,0.0006,0.9135,0.0080,0.8935,0.9335,2242,43.9141,0.0000
2,bottleneck_mlp,3,0.9497,0.0017,0.9130,0.0040,0.9030,0.9231,2262,47.5865,0.0000
3,vqc_q4_l2_cyclic,3,0.9387,0.0007,0.9036,0.0045,0.8925,0.9147,2266,62.7204,8.3123
4,vqc_q4_l2_noent,3,0.9459,0.0037,0.9079,0.0055,0.8942,0.9215,2266,98.3354,18.4906
5,vqc_q4_l1_cyclic,3,0.9477,0.0069,0.9076,0.0052,0.8948,0.9204,2254,66.5924,7.7118
6,vqc_q4_l4_cyclic,3,0.9442,0.0029,0.9056,0.0056,0.8916,0.9196,2290,128.8871,31.4042
7,vqc_q2_l2_cyclic,3,0.9065,0.0245,0.8754,0.0260,0.8108,0.9399,1192,50.4447,5.3018
8,vqc_q8_l2_cyclic,3,0.9403,0.0017,0.9042,0.0011,0.9015,0.9069,4414,114.4209,27.5543
9,bottleneck_mlp_k8,3,0.9501,0.0034,0.9104,0.0064,0.8945,0.9262,4438,48.6753,0.0000


,comparison,seeds_n,seeds_mean_difference,seeds_t_stat,seeds_p_ttest,seeds_p_wilcoxon,boot_metric,boot_difference,boot_ci_low,boot_ci_high,boot_p_value,boot_n_valid,boot_p_holm
0,vqc_q4_l2_cyclic - bottleneck_mlp,3,-0.0094,-2.1624,0.1631,0.25,macro_auc,-0.0083,-0.0140,-0.0029,0.008,500,0.024
1,vqc_q4_l2_cyclic - bottleneck_identity,3,-0.0099,-1.4399,0.2866,0.50,macro_auc,-0.0099,-0.0202,-0.0027,0.024,500,0.048
2,vqc_q4_l2_cyclic - linear_probe,3,-0.0112,-4.9558,0.0384,0.25,macro_auc,-0.0075,-0.0238,0.0161,0.376,500,0.376


PosixPath('/kaggle/working/rsna_ich_artifacts/pilot/analysis/trackC_paired.csv')

## 6. Result statement generated from the numbers above (no pre-written conclusion)

In [11]:
def statement(row):
    d, lo, hi = row.get("boot_difference", np.nan), row.get("boot_ci_low", np.nan), row.get("boot_ci_high", np.nan)
    if not np.isfinite(d):
        return f"{row['comparison']}: not evaluable (missing runs)."
    if lo > 0:
        verdict = "the quantum head is better, and the 95% patient-level CI excludes 0"
    elif hi < 0:
        verdict = "the quantum head is WORSE, and the 95% patient-level CI excludes 0"
    else:
        verdict = "no statistically reliable difference (CI includes 0)"
    return f"{row['comparison']}: delta macro AUC = {d:+.4f} [{lo:+.4f}, {hi:+.4f}] -> {verdict}. MODE={MODE}."
for _, r in cmp_df.iterrows():
    print(statement(r))
print("Simulated, noise-free circuit on classical hardware; no claim about quantum hardware or speed-up is made.")

vqc_q4_l2_cyclic - bottleneck_mlp: delta macro AUC = -0.0083 [-0.0140, -0.0029] -> the quantum head is WORSE, and the 95% patient-level CI excludes 0. MODE=pilot.
vqc_q4_l2_cyclic - bottleneck_identity: delta macro AUC = -0.0099 [-0.0202, -0.0027] -> the quantum head is WORSE, and the 95% patient-level CI excludes 0. MODE=pilot.
vqc_q4_l2_cyclic - linear_probe: delta macro AUC = -0.0075 [-0.0238, +0.0161] -> no statistically reliable difference (CI includes 0). MODE=pilot.
Simulated, noise-free circuit on classical hardware; no claim about quantum hardware or speed-up is made.


## 7. Trainability and cost diagnostics

In [12]:
from src.models.hybrid_classifier import FeatureHead
from src.quantum import angle_saturation
diag = []
for spec in grid:
    if spec["kind"] == "vqc":
        d = ctx.dir("experiments", f"C_{spec['id']}_s{qcfg['seeds'][0]}")
        if (d / "history.csv").exists():
            h = pd.read_csv(d / "history.csv")
            diag.append({"head": spec["id"], "mean_grad_norm": h["grad_norm"].mean(), "last_grad_norm": h["grad_norm"].iloc[-1],
                         "epochs": len(h)})
display(pd.DataFrame(diag))
hd = head_cfg(qcfg["primary"][-1]); head = FeatureHead(in_dim=feats["train"]["features"].shape[1], **hd)
from src.training.checkpointing import load_checkpoint
ck = ctx.dir("experiments", f"C_{qcfg['primary'][-1]['id']}_s{qcfg['seeds'][0]}") / "best.pt"
if ck.exists():
    load_checkpoint(ck, head, expected_manifest_hash=man_info["manifest_hash"])
    print("Angle saturation on validation features:", angle_saturation(head, torch.as_tensor(feats["val"]["features"][:4096])))

,head,mean_grad_norm,last_grad_norm,epochs
0,vqc_q4_l2_cyclic,0.191379,0.163096,9
1,vqc_q4_l2_noent,0.190885,0.222310,16
2,vqc_q4_l1_cyclic,0.207213,0.233445,16
3,vqc_q4_l4_cyclic,0.201117,0.251863,22
4,vqc_q2_l2_cyclic,0.451207,0.163274,11
5,vqc_q8_l2_cyclic,0.180835,0.241919,15


Angle saturation on validation features: {'mean_abs_angle_over_pi': 0.5155586004257202, 'frac_saturated': 0.027099609375}
